In [20]:
import pandas as pd
import re
original_df = pd.read_json("synthetic_incidents.jsonl", lines=True)

original_df.head()

,id,created_at,scenario_id,run_id,namespace,workload_name,image,real_pod_name,container_name,log_container_name,manifest_path,pod_describe,pod_logs,pod_logs_previous,deployment_yaml,apply_result
0,0f6ec139-c95d-429b-921c-dbfc98e82df4,2026-03-31 06:37:40.393570+00:00,pvc_not_found_mountfail,pn3zirfh,data-pipeline,busybox-pn3zirfh,busybox:1.36,busybox-pn3zirfh-86bbb7957c-7754l,app,app,k8s_out/pvc_not_found_mountfail-busybox-pn3zirfh-data-pipeline-pn3zirfh.yaml,Name: busybox-pn3zirfh-86bbb7957c-7754l\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-pn3zirfh\n pod-template-hash=86bbb7957c\n run_id=pn3zirfh\n ...,,,"apiVersion: apps/v1\nkind: Deployment\nmetadata:\n annotations:\n deployment.kubernetes.io/revision: ""1""\n kubectl.kubernetes.io/last-applied-configuration: |\n {""apiVersion"":""apps/v1"",""kind"":""Deployment"",""metadata"":{""annotations"":{},""labels"":{""app"":""busybox-pn3zirfh"",""run_id"":""pn3zi...","{'cmd': 'kubectl apply -f k8s_out/pvc_not_found_mountfail-busybox-pn3zirfh-data-pipeline-pn3zirfh.yaml', 'returncode': 0, 'stdout': 'deployment.apps/busybox-pn3zirfh created ', 'stderr': ''}"
1,70607b4c-0478-4991-b8ff-c8e04c86b657,2026-03-31 06:37:45.527867+00:00,rbac_forbidden,hnw5520p,monitoring,kube-state-metrics-hnw5520p,registry.k8s.io/kube-state-metrics/kube-state-metrics:v2.12.0,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app,app,k8s_out/rbac_forbidden-kube-state-metrics-hnw5520p-monitoring-hnw5520p.yaml,"Name: kube-state-metrics-hnw5520p-7fbb7866df-8rsgh\nNamespace: monitoring\nPriority: 0\nService Account: app-sa-hnw5520p\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-state-metrics-hnw55...","I0331 06:37:30.545502 1 wrapper.go:120] ""Starting kube-state-metrics""\nW0331 06:37:30.545690 1 client_config.go:618] Neither --kubeconfig nor --master was specified. Using the inClusterConfig. This might not work.\nI0331 06:37:30.545773 1 server.go:199] ""Used resources"" resou...","Error from server (BadRequest): previous terminated container ""app"" in pod ""kube-state-metrics-hnw5520p-7fbb7866df-8rsgh"" not found\n","apiVersion: apps/v1\nkind: Deployment\nmetadata:\n annotations:\n deployment.kubernetes.io/revision: ""1""\n kubectl.kubernetes.io/last-applied-configuration: |\n {""apiVersion"":""apps/v1"",""kind"":""Deployment"",""metadata"":{""annotations"":{},""labels"":{""app"":""kube-state-metrics-hnw5520p"",""run...","{'cmd': 'kubectl apply -f k8s_out/rbac_forbidden-kube-state-metrics-hnw5520p-monitoring-hnw5520p.yaml', 'returncode': 0, 'stdout': 'serviceaccount/app-sa-hnw5520p created deployment.apps/kube-state-metrics-hnw5520p created ', 'stderr': ''}"
2,f2406c58-17ea-4a01-8638-62d04ccf9c54,2026-03-31 06:37:45.528651+00:00,rbac_forbidden,4g3nkfre,qa-app,ingress-nginx-4g3nkfre,registry.k8s.io/ingress-nginx/controller:v1.13.3,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app,app,k8s_out/rbac_forbidden-ingress-nginx-4g3nkfre-qa-app-4g3nkfre.yaml,"Name: ingress-nginx-4g3nkfre-656c6b79b6-vbbbn\nNamespace: qa-app\nPriority: 0\nService Account: app-sa-4g3nkfre\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-4g3nkfre\n ...",-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,"apiVersion: apps/v1\nkind: Deployment\nmetadata:\n annotations:\n deployment.kubernetes.io/revision: ""1""\n kubectl.kubernetes.io/last-applied-configuration: |\n {""apiVersion"":""apps/v

In [21]:
df = original_df[["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]].copy()

In [22]:
df.head()

,scenario_id,pod_describe,pod_logs,pod_logs_previous
0,pvc_not_found_mountfail,Name: busybox-pn3zirfh-86bbb7957c-7754l\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-pn3zirfh\n pod-template-hash=86bbb7957c\n run_id=pn3zirfh\n ...,,
1,rbac_forbidden,"Name: kube-state-metrics-hnw5520p-7fbb7866df-8rsgh\nNamespace: monitoring\nPriority: 0\nService Account: app-sa-hnw5520p\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-state-metrics-hnw55...","I0331 06:37:30.545502 1 wrapper.go:120] ""Starting kube-state-metrics""\nW0331 06:37:30.545690 1 client_config.go:618] Neither --kubeconfig nor --master was specified. Using the inClusterConfig. This might not work.\nI0331 06:37:30.545773 1 server.go:199] ""Used resources"" resou...","Error from server (BadRequest): previous terminated container ""app"" in pod ""kube-state-metrics-hnw5520p-7fbb7866df-8rsgh"" not found\n"
2,rbac_forbidden,"Name: ingress-nginx-4g3nkfre-656c6b79b6-vbbbn\nNamespace: qa-app\nPriority: 0\nService Account: app-sa-4g3nkfre\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-4g3nkfre\n ...",-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...
3,rbac_forbidden,"Name: ingress-nginx-n5oblcdc-78d4488dfd-gnt5g\nNamespace: prod-app\nPriority: 0\nService Account: app-sa-n5oblcdc\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-n5oblcdc\n ...",-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...
4,rbac_forbidden,"Name: kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4\nNamespace: deployment\nPriority: 0\nService Account: app-sa-yrnzw8kt\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-state-metrics-yrnzw...","I0331 06:37:30.542740 1 wrapper.go:120] ""Starting kube-state-metrics""\nW0331 06:37:30.542943 1 client_config.go:618] Neither --kubeconfig nor --master was specified. Using the inClusterConfig. This might not work.\nI0331 06:37:30.543061 1 server.go:199] ""Used resources"" resou...","Error from server (BadRequest): previous terminated container ""app"" in pod ""kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4"" not found\n"


In [23]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

In [24]:
for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    df[col] = df[col].apply(normalize_text)

In [25]:
def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

In [26]:
def extract_block(text, block_name):
    if not text:
        return None

    lines = text.split("\n")
    block = []
    in_block = False

    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue

        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break

        block.append(line)

    out = "\n".join(block).strip()
    return out if out else None

In [27]:
def parse_events_block(events_text):
    if not events_text:
        return {
            "event_reason": None,
            "event_message": None,
            "all_event_reasons": [],
            "all_event_messages": []
        }

    reasons = []
    messages = []

    lines = [line for line in events_text.split("\n") if line.strip()]

    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue

        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())

    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
        "all_event_reasons": reasons,
        "all_event_messages": messages
    }

In [28]:
def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    tolerations_block = extract_block(text, "Tolerations")
    events_block = extract_block(text, "Events")

    events = parse_events_block(events_block)

    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "pod_ip": extract_first(r"^IP:\s*(.+)$", text),
        "controlled_by": extract_first(r"^Controlled By:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "qos_class": extract_first(r"^QoS Class:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "describe_events_raw": events_block,
        "describe_containers_raw": containers_block,
        "describe_volumes_raw": volumes_block,
        "describe_tolerations_raw": tolerations_block,
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
        "all_event_reasons": events["all_event_reasons"],
        "all_event_messages": events["all_event_messages"],
    }

In [29]:
describe_parsed = df["pod_describe"].apply(parse_describe).apply(pd.Series)
df = pd.concat([df, describe_parsed], axis=1)

In [30]:
def build_evidence_text(row):
    parts = []

    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])

    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])

    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])

    return "\n\n".join(parts).strip() if parts else None

In [31]:
df["evidence_text"] = df.apply(build_evidence_text, axis=1)

In [32]:
parsed_df = df[
    [
        "scenario_id",
        "namespace",
        "pod_name",
        "service_account_name",
        "node",
        "pod_status",
        "image",
        "container_state",
        "last_state",
        "ready",
        "restart_count",
        "node_selectors",
        "claim_name",
        "event_reason",
        "event_message",
        "evidence_text",
    ]
].copy()

In [34]:
parsed_df.head(20)

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,NaN,NaN,NaN,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,"0/1 nodes are available: persistentvolumeclaim ""missing-pvc-pn3zirfh"" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",POD DESCRIBE:\nName: busybox-pn3zirfh-86bbb7957c-7754l\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-pn3zirfh\n pod-template-hash=86bbb7957c\n run_id=pn3zirfh...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-metrics:v2.12.0,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned monitoring/kube-state-metrics-hnw5520p-7fbb7866df-8rsgh to kind-control-plane,"POD DESCRIBE:\nName: kube-state-metrics-hnw5520p-7fbb7866df-8rsgh\nNamespace: monitoring\nPriority: 0\nService Account: app-sa-hnw5520p\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-stat..."
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3nkfre-656c6b79b6-vbbbn to kind-control-plane,"POD DESCRIBE:\nName: ingress-nginx-4g3nkfre-656c6b79b6-vbbbn\nNamespace: qa-app\nPriority: 0\nService Account: app-sa-4g3nkfre\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-4g3n..."
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned prod-app/ingress-nginx-n5oblcdc-78d4488dfd-gnt5g to kind-control-plane,"POD DESCRIBE:\nName: ingress-nginx-n5oblcdc-78d4488dfd-gnt5g\nNamespace: prod-app\nPriority: 0\nService Account: app-sa-n5oblcdc\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-n5..."
4,rbac_forbidden,deployment,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app-sa-yrnzw8kt,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-metrics:v2.12.0,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned deployment/kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4 to kind-control-plane,"POD DESCRIBE:\nName: kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4\nNamespace: deployment\nPriority: 0\nService Account: app-sa-yrnzw8kt\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-stat..."
5,pvc_not_found_mountfail,app,redis-wuwdazqz-6cbddc486f-dt8vw,default,<none>,Pending,redis:7.2,NaN,NaN,NaN,NaN,<none>,missing-pvc-wuwdazqz,FailedScheduling,"0/1 nodes are available: persistentvolumeclaim ""missing-pvc-wuwdazqz"" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",POD DESCRIBE:\nName: redis-wuwdazqz-6cbddc486f-dt8vw\nNamespace: app\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=redis-wuwdazqz\n pod-template-hash=6cbddc486f\n run_id=wuwdazqz\n ...
6,pvc_not_found_mountfail,monitoring,nginx-0kbrd6uu-65c96d8888-5qkdp,default,<none>,Pending,nginx:1.27,NaN,NaN,NaN,NaN,<none>,missing-pvc-0kbrd6uu,FailedScheduling,"0/1 nodes are available: persistentvolumeclaim ""missing-pvc-0kbrd6uu"" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",POD DESCRIBE:\nName: nginx-0kbrd6uu-65c96d8888-5qkdp\nNamespace: monitoring\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=n

In [37]:
preview_df = sample_df.copy()

for col in ["event_message", "evidence_text"]:
    if col in preview_df.columns:
        preview_df[col] = preview_df[col].fillna("").str.slice(0, 200)

preview_df

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,evidence_text
0,failedscheduling_taint,qa-app,nginx-n0qpqzfi-bb4f4dcb8-tpthp,default,<none>,Pending,nginx:1.27,NaN,NaN,NaN,NaN,dedicated=gpu,NaN,FailedScheduling,0/1 nodes are available: 1 node(s) had untolerated taint {dedicated: gpu}. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.,POD DESCRIBE:\nName: nginx-n0qpqzfi-bb4f4dcb8-tpthp\nNamespace: qa-app\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=nginx-n0qpqzfi\n
1,pvc_not_found_mountfail,data-pipeline,busybox-xftu96pz-dd646b45f-w75hf,default,<none>,Pending,busybox:1.36,NaN,NaN,NaN,NaN,<none>,missing-pvc-xftu96pz,FailedScheduling,"0/1 nodes are available: persistentvolumeclaim ""missing-pvc-xftu96pz"" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",POD DESCRIBE:\nName: busybox-xftu96pz-dd646b45f-w75hf\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-xf
2,rbac_forbidden,data-pipeline,ingress-nginx-hqe73279-6596484cf4-r557r,app-sa-hqe73279,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Terminated,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned data-pipeline/ingress-nginx-hqe73279-6596484cf4-r557r to kind-control-plane,POD DESCRIBE:\nName: ingress-nginx-hqe73279-6596484cf4-r557r\nNamespace: data-pipeline\nPriority: 0\nService Account: app-sa-hqe73279\nNode: kind-control-plane/172.1


In [38]:
final_df = df.copy()

In [39]:
cols = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

In [40]:
final_df = df[cols].copy()
final_df.head()

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,pod_describe,pod_logs,pod_logs_previous,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,NaN,NaN,NaN,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,"0/1 nodes are available: persistentvolumeclaim ""missing-pvc-pn3zirfh"" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",Name: busybox-pn3zirfh-86bbb7957c-7754l\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-pn3zirfh\n pod-template-hash=86bbb7957c\n run_id=pn3zirfh\n ...,,,POD DESCRIBE:\nName: busybox-pn3zirfh-86bbb7957c-7754l\nNamespace: data-pipeline\nPriority: 0\nService Account: default\nNode: <none>\nLabels: app=busybox-pn3zirfh\n pod-template-hash=86bbb7957c\n run_id=pn3zirfh...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-metrics:v2.12.0,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned monitoring/kube-state-metrics-hnw5520p-7fbb7866df-8rsgh to kind-control-plane,"Name: kube-state-metrics-hnw5520p-7fbb7866df-8rsgh\nNamespace: monitoring\nPriority: 0\nService Account: app-sa-hnw5520p\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-state-metrics-hnw55...","I0331 06:37:30.545502 1 wrapper.go:120] ""Starting kube-state-metrics""\nW0331 06:37:30.545690 1 client_config.go:618] Neither --kubeconfig nor --master was specified. Using the inClusterConfig. This might not work.\nI0331 06:37:30.545773 1 server.go:199] ""Used resources"" resou...","Error from server (BadRequest): previous terminated container ""app"" in pod ""kube-state-metrics-hnw5520p-7fbb7866df-8rsgh"" not found","POD DESCRIBE:\nName: kube-state-metrics-hnw5520p-7fbb7866df-8rsgh\nNamespace: monitoring\nPriority: 0\nService Account: app-sa-hnw5520p\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=kube-stat..."
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3nkfre-656c6b79b6-vbbbn to kind-control-plane,"Name: ingress-nginx-4g3nkfre-656c6b79b6-vbbbn\nNamespace: qa-app\nPriority: 0\nService Account: app-sa-4g3nkfre\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-4g3nkfre\n ...",-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,-------------------------------------------------------------------------------\nNGINX Ingress controller\n Release: v1.13.3\n Build: 93851f05e61d99eea49140c9be73499a3cb92ccc\n Repository: https://github.com/kubernetes/ingress-nginx\n nginx version: nginx/1.27.1\n\n---------...,"POD DESCRIBE:\nName: ingress-nginx-4g3nkfre-656c6b79b6-vbbbn\nNamespace: qa-app\nPriority: 0\nService Account: app-sa-4g3nkfre\nNode: kind-control-plane/172.18.0.5\nStart Time: Mon, 30 Mar 2026 23:37:29 -0700\nLabels: app=ingress-nginx-4g3n..."
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned prod-app/ingress-nginx-n5oblcdc-78d4488dfd-gnt5g to kind-control-plane,"Name: ingress-nginx-n5oblcdc-78d4488dfd-gnt5g\nNamespace: prod-app\nPriority: 0\nService Account: app-sa-n5o